# DeepSAD and ESAD

When a few anomalies are already known, semi-supervised detectors can use them. **DeepSAD** pretrains an autoencoder and then pulls normal data towards a centre in latent space while pushing the labelled anomalies away. **ESAD** trains everything in one stage. Both then rank the **unlabelled** points.

This tutorial needs OutlierDetectionNetworks.jl, OutlierDetection.jl and Flux.jl.

In [ ]:
using OutlierDetection, OutlierDetectionNetworks, Flux
using OutlierDetectionNetworks.Templates: MLPAutoEncoder
using Plots, Statistics, Random

Random.seed!(1);

## Data and Labels

Three hundred normal points and fifteen anomalies from a shifted cloud. We know only five of the anomalies. All other rows are labelled "normal", which is also how the methods treat unlabelled data.

In [ ]:
X = vcat(randn(300, 4), 3 .+ randn(15, 4))
is_true = vcat(falses(300), trues(15))
anom_idx = 301:305   # the known anomalies

y = fill("normal", size(X, 1))
y[anom_idx] .= "outlier"

lo, hi = minimum(X, dims = 1), maximum(X, dims = 1)
Xs = Float32.(permutedims((X .- lo) ./ (hi .- lo)))
unknown = setdiff(findall(is_true), anom_idx)   # the anomalies the detectors must find

## DeepSAD

`epochs = (10, 30)` means 10 pretraining epochs followed by 30 training epochs. `eta` weights the labelled points.

In [ ]:
encoder, decoder = MLPAutoEncoder(size(Xs, 1), 2, [32, 16]; bias = true)
dsad = DSADDetector(encoder = encoder, decoder = decoder, epochs = (10, 30), eta = 1)
_, dsad_scores = OutlierDetection.fit(dsad, Xs, to_categorical(y); verbosity = 0)

## ESAD

In [ ]:
encoder, decoder = MLPAutoEncoder(size(Xs, 1), 2, [32, 16]; bias = true)
esad = ESADDetector(encoder = encoder, decoder = decoder, epochs = 30)
_, esad_scores = OutlierDetection.fit(esad, Xs, to_categorical(y); verbosity = 0)

## Did They Find the Unknown Anomalies?

The labelled anomalies score high by construction, so we judge the detectors on the ten **unlabelled** anomalies. For comparison, an unsupervised autoencoder is fitted on the same data.

In [ ]:
encoder, decoder = MLPAutoEncoder(size(Xs, 1), 2, [32, 16]; bias = true)
_, ae_scores = OutlierDetection.fit(AEDetector(encoder = encoder, decoder = decoder, epochs = 30), Xs; verbosity = 0)

found(s) = count(in(unknown), sortperm(s; rev = true)[1:20])   # unknown anomalies in the top 20
for (name, s) in [("AutoEncoder", ae_scores), ("DeepSAD", dsad_scores), ("ESAD", esad_scores)]
    println(rpad(name, 12), found(s), " of ", length(unknown), " unknown anomalies in the top 20")
end

In [ ]:
p1 = scatter(X[:, 1], X[:, 2]; zcolor = dsad_scores, markerstrokewidth = 0, title = "DeepSAD", label = "")
p2 = scatter(X[:, 1], X[:, 2]; zcolor = esad_scores, markerstrokewidth = 0, title = "ESAD", label = "")
plot(p1, p2; layout = (1, 2), size = (900, 380))

Labels are not guaranteed to help. Here the anomalies form a well-separated cloud, and the unsupervised autoencoder already finds them. The semi-supervised methods pay off when anomalies are subtle and resemble the labelled examples; always keep an unsupervised baseline.

## Agreement

The two semi-supervised detectors optimise different objectives, so their ordering of ordinary points differs. Compare their top lists instead of the full ranking.

In [ ]:
top20(s) = sortperm(s; rev = true)[1:20]
println("Shared points in the DeepSAD and ESAD top-20 lists: ", length(intersect(top20(dsad_scores), top20(esad_scores))))

## Summary
- DeepSAD and ESAD use a few labelled anomalies to sharpen the detector.
- Label known anomalies "outlier" and everything else "normal".
- Judge them on the unlabelled points and against an unsupervised baseline.
- ESAD divides by the latent distance; keep `bias = true` and hidden layers of at least 16 units to avoid a NaN loss.